<a href="https://colab.research.google.com/github/Dharshini11042004/Adaptive-Threat-Detection-in-Smart-Healthcare-Infrastructure-with-Agentic-AI-using-RL/blob/main/Data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os
zip_path = "/content/Datasets.zip"
extract_path = "/content"
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print("Extraction complete")
import re
def classify_column(col_name):
    col = col_name.lower()
    # IDENTIFIERS
    if any(x in col for x in ["flow id","ip","timestamp","id"]):
        return "IDENTIFIER"
    # NETWORK METADATA
    if any(x in col for x in [
        "flow","duration","rate","header","tcp","udp",
        "http","https","dns","flag","syn","ack","fin",
        "rst","iat","bytes","sum","min","max","avg","std",
        "packet","frame","length","protocol"
    ]):
        return "NETWORK_METADATA"
    # IOT / MODBUS RELATED
    if any(x in col for x in [
        "device","sensor","firmware","uptime","battery","iot",
        "modbus","function","register","coil",
        "holding","input","transaction"
    ]):
        return "IOT_BEHAVIOR"
    # LINUX SYSTEM METRICS
    if any(x in col for x in [
        "cpu","memory","disk","process","login","thread",
        "cache","swap","rss","virtual","heap"
    ]):
        return "HOST_SYSTEM"
    # HEALTHCARE APP
    if any(x in col for x in [
        "api","role","access","auth","session"
    ]):
        return "HEALTHCARE_APP_LOG"
    # IMPORTANT FIX:
    # Keep unknown numeric telemetry columns instead of discarding them
    return "KEEP"
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
def process_dataset(file_path):
    print(f"\nProcessing: {file_path}")
    df = pd.read_csv(file_path, low_memory=False)
    # ---------- Column Classification ----------
    column_types = {col: classify_column(col) for col in df.columns}
    # Keep allowed healthcare-accessible data
    allowed_types = [
        "NETWORK_METADECATA",
        "IOT_BEHAVIOR",
        "HOST_SYSTEM",
        "HEALTHCARE_APP_LOG",
        "KEEP"  # <-- NEW: prevents empty datasets
    ]
    # Use a set to ensure unique column names
    allowed_columns_set = set(
        col for col,ctype in column_types.items()
        if ctype in allowed_types
    )
    # Always retain Label if present
    if "Label" in df.columns:
        allowed_columns_set.add("Label")
    df = df[list(allowed_columns_set)] # Convert set back to list for DataFrame indexing
    # Safety check to prevent empty dataset
    if df.shape[1] == 0:
        print("Warning: No usable columns after filtering.")
        return pd.DataFrame(), pd.DataFrame(), column_types
    # ---------- Clean ----------
    df = df.dropna()
    df = df.loc[:, df.nunique() > 1]
    # ---------- Correlation Filtering ----------
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = numeric_cols.drop("Label", errors="ignore")
    if len(numeric_cols) > 1:
        corr = df[numeric_cols].corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
        df = df.drop(columns=to_drop)
    # ---------- Zero-Day Split ----------
    if "Label" in df.columns and df.shape[0] > 10:
        normal_class = df["Label"].value_counts().idxmax()
        attack_types = df[df["Label"] != normal_class]["Label"].unique()
        if len(attack_types) > 1:
            np.random.seed(42)
            num_zero = max(1, int(0.1 * len(attack_types)))
            zero_day = np.random.choice(attack_types, num_zero, replace=False)
            zero_df = df[df["Label"].isin(zero_day)]
            remain_df = df[~df["Label"].isin(zero_day)]
            train_df, test_partial = train_test_split(
                remain_df,
                test_size=0.3,
                stratify=remain_df["Label"],
                random_state=42
            )
            test_df = pd.concat([test_partial, zero_df])
        else:
            train_df, test_df = train_test_split(
                df,
                test_size=0.3,
                stratify=df["Label"],
                random_state=42
            )
    else:
        train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
    # ---------- Safe Normalization ----------
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns
    numeric_cols = numeric_cols.drop("Label", errors="ignore")
    if len(numeric_cols) > 0:
        mean = train_df[numeric_cols].mean()
        std = train_df[numeric_cols].std().replace(0,1)
        train_df[numeric_cols] = (train_df[numeric_cols] - mean) / std
        test_df[numeric_cols] = (test_df[numeric_cols] - mean) / std
    return train_df, test_df, column_types
import shutil
processed_output = "/content/processed_agents"
# Clean previous runs
if os.path.exists(processed_output):
    shutil.rmtree(processed_output)
os.makedirs(processed_output, exist_ok=True)
csv_files = []
for root, dirs, files in os.walk(extract_path):
    if root.startswith(processed_output):
        continue
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))
print("Total CSV files found:", len(csv_files))
for file_path in csv_files:
    train_df, test_df, col_map = process_dataset(file_path)
    if train_df.empty or test_df.empty:
        print("Skipped empty dataset:", file_path)
        continue
    base = os.path.basename(file_path).replace(".csv","")
    train_df.to_csv(f"{processed_output}/{base}_train.csv", index=False)
    test_df.to_csv(f"{processed_output}/{base}_test.csv", index=False)
print("Processed files:", os.listdir(processed_output))
final_zip = "/content/Agent0_intelligent_processed.zip"
with zipfile.ZipFile(final_zip, 'w') as z:
    for file in os.listdir(processed_output):
        z.write(os.path.join(processed_output,file), arcname=file)
print("Final ZIP created:", final_zip)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Datasets.zip'